In [3]:
import numpy as np
import pandas as pd
import os

# --- CONFIG ---
BASE_PATH = "/content/drive/MyDrive/1 Skripsi/new approach v2/"

# PATHS TO YOUR "VPN ONLY" OUTPUTS
VPN_ALPHA_NPY = os.path.join(BASE_PATH, "VPNOnly-cnn_payload_data.npy")
VPN_ALPHA_LABELS = os.path.join(BASE_PATH, "VPNOnly-cnn_payload_labels.csv")
VPN_BETA_CSV = os.path.join(BASE_PATH, "VPNOnly-delta_component_v2.csv")
VPN_GAMMA_CSV = os.path.join(BASE_PATH, "VPNOnly-gamma_prime_component_v2.csv")
VPN_FFT_CSV = os.path.join(BASE_PATH, "VPNOnly-fft_component_v2.csv")

# PATHS TO YOUR "NON-VPN / FULL" OUTPUTS
# (Update these names to match what you actually saved them as)
FULL_ALPHA_NPY = os.path.join(BASE_PATH, "cnn_payload_data.npy")
FULL_ALPHA_LABELS = os.path.join(BASE_PATH, "cnn_payload_labels.csv")
FULL_BETA_CSV = os.path.join(BASE_PATH, "delta_component_merged_v2.csv")
FULL_GAMMA_CSV = os.path.join(BASE_PATH, "gamma_prime_component_v2.csv")
FULL_FFT_CSV = os.path.join(BASE_PATH, "fft_component_v2.csv")

# OUTPUT FOR PRE-TRAINING
OUT_VIEW1 = os.path.join(BASE_PATH, "PRETRAIN_X_view1.npy")
OUT_VIEW2 = os.path.join(BASE_PATH, "PRETRAIN_X_view2.npy")

def load_and_align(alpha_npy, alpha_csv, beta, gamma, fft):
    print(f"Loading {alpha_csv}...")
    X_alpha = np.load(alpha_npy)
    df_alpha = pd.read_csv(alpha_csv)
    df_alpha['original_index'] = df_alpha.index

    # Load Stats
    df_beta = pd.read_csv(beta)
    df_gamma = pd.read_csv(gamma)
    df_fft = pd.read_csv(fft)

    # Merge Stats
    df_stats = pd.merge(df_beta, df_gamma, on="filename", suffixes=('_beta', '_gamma'))
    df_stats = pd.merge(df_stats, df_fft, on="filename")

    # Align with Alpha
    df_merged = pd.merge(df_alpha, df_stats, on="filename", how="inner")

    # Extract
    valid_indices = df_merged['original_index'].values
    X_view1 = X_alpha[valid_indices]

    drop_cols = [c for c in df_merged.columns if isinstance(c, str) and
                 any(x in c for x in ['filename', 'application', 'category', 'binary_type', 'original_index'])]

    numeric_cols = [c for c in df_merged.columns if c not in drop_cols and df_merged[c].dtype in ['float64', 'int64']]
    X_view2 = df_merged[numeric_cols].values

    return X_view1, X_view2

def main():
    # 1. Process VPN Data
    print("--- Processing VPN Data ---")
    X1_vpn, X2_vpn = load_and_align(VPN_ALPHA_NPY, VPN_ALPHA_LABELS, VPN_BETA_CSV, VPN_GAMMA_CSV, VPN_FFT_CSV)

    # 2. Process Non-VPN Data
    print("--- Processing Non-VPN Data ---")
    # Only run this if the files exist
    if os.path.exists(FULL_ALPHA_NPY):
        X1_full, X2_full = load_and_align(FULL_ALPHA_NPY, FULL_ALPHA_LABELS, FULL_BETA_CSV, FULL_GAMMA_CSV, FULL_FFT_CSV)

        # 3. CONCATENATE (The Virtual Merge)
        print("--- Stacking Datasets ---")
        X1_final = np.concatenate([X1_vpn, X1_full], axis=0)
        X2_final = np.concatenate([X2_vpn, X2_full], axis=0)
    else:
        print("WARNING: Non-VPN files not found. Using VPN-Only for Pre-training (Sub-optimal).")
        X1_final = X1_vpn
        X2_final = X2_vpn

    print(f"Final Pre-training Size: {X1_final.shape[0]} samples")

    # 4. Save
    np.save(OUT_VIEW1, X1_final)
    np.save(OUT_VIEW2, X2_final)
    print("Done. Use these files for Step 02 (Pre-training).")

if __name__ == "__main__":
    main()

--- Processing VPN Data ---
Loading /content/drive/MyDrive/1 Skripsi/new approach v2/VPNOnly-cnn_payload_labels.csv...
--- Processing Non-VPN Data ---
Loading /content/drive/MyDrive/1 Skripsi/new approach v2/cnn_payload_labels.csv...
--- Stacking Datasets ---
Final Pre-training Size: 12165 samples
Done. Use these files for Step 02 (Pre-training).
